# Pipeline Validation Tests

Data quality and consistency tests for the taxi-trip medallion pipeline.

**What is tested**

| # | Test | Layer | Guard against |
|---|------|-------|---------------|
| 1 | Silver row count does not exceed raw trips count | Bronze → Silver | Duplicated ingestion |
| 2 | Silver has no null/negative `total_amount` or non-positive `trip_distance` | Silver | Bad records passing the quality check |
| 3 | Silver has no null pickup/dropoff timestamps | Silver | Unparseable datetime strings |
| 4 | `taxi_zone_revenue_gold` trip count reconciles with silver | Gold | Broken group-by/join in gold aggregation |
| 5 | `taxi_peak_hours_gold` trip count reconciles with silver | Gold | Missing hour buckets |
| 6 | `taxi_time_of_day_revenue_gold` trip count reconciles with silver | Gold | Bucket overlap or gap in time-of-day CASE buckets |

Run all cells. Each test prints `PASS` or `FAIL`; the final cell raises an `AssertionError` if any test failed, so the notebook can be used as a job task or CI gate.

In [0]:
RAW_TRIPS = "workspace.taxi_trip.trips"
RAW_ZONES = "workspace.taxi_trip.zones"
SILVER = "workspace.default.taxi_trips_silver"
GOLD_ZONE_REVENUE = "workspace.default.taxi_zone_revenue_gold"
GOLD_PEAK_HOURS = "workspace.default.taxi_peak_hours_gold"
GOLD_TIME_OF_DAY = "workspace.default.taxi_time_of_day_revenue_gold"

In [0]:
def check(name, ok):
    """Record a test result and print PASS/FAIL."""
    results.append((name, bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

results = []

In [0]:
# Test 1: silver never contains more rows than raw (guards against duplicate ingestion)
raw_count = spark.table(RAW_TRIPS).count()
silver_count = spark.table(SILVER).count()
check("silver_count_le_raw_count", silver_count <= raw_count)
check("silver_retention_rate", 0 < silver_count / raw_count <= 1)
print(f"raw={raw_count}, silver={silver_count}, retention={silver_count / raw_count:.1%}")

In [0]:
# Test 2: silver has no null/negative amounts or non-positive distances
check(
    "silver_no_invalid_amount_or_distance",
    spark.table(SILVER).where("total_amount IS NULL OR total_amount <= 0 OR trip_distance <= 0").count() == 0,
)

# Test 3: silver has no null pickup/dropoff timestamps
check(
    "silver_no_null_timestamps",
    spark.table(SILVER).where("pickup_time IS NULL OR dropoff_time IS NULL").count() == 0,
)

# Zones dimension: every row must have a LocationID (mirrors the DLT expectation)
check(
    "zones_have_no_null_location_id",
    spark.table(RAW_ZONES).where("LocationID IS NULL").count() == 0,
)

In [0]:
# Tests 4-6: gold aggregations reconcile with silver (no lost or duplicated trips)
check(
    "gold_zone_revenue_reconciles",
    spark.sql(f"SELECT SUM(total_trips) FROM {GOLD_ZONE_REVENUE}").first()[0] == silver_count,
)
check(
    "gold_peak_hours_reconciles",
    spark.sql(f"SELECT SUM(total_trips) FROM {GOLD_PEAK_HOURS}").first()[0] == silver_count,
)
check(
    "gold_time_of_day_reconciles",
    spark.sql(f"SELECT SUM(total_trips) FROM {GOLD_TIME_OF_DAY}").first()[0] == silver_count,
)

In [0]:
# Summary — raises if any test failed, so this notebook can gate a job or CI run
print(f"\n{sum(ok for _, ok in results)}/{len(results)} tests passed")
failed = [name for name, ok in results if not ok]
assert not failed, f"Validation failed: {failed}"